In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys

PROJECT_DIR = "/content/drive/MyDrive/BehavioralAI"

sys.path.append(PROJECT_DIR)

In [3]:
DATA_DIR = "/content/drive/MyDrive/BehavioralAI/data"

MODELS_DIR = "/content/drive/MyDrive/BehavioralAI/models"

ACCESS_LOGS = f"{DATA_DIR}/access_logs.parquet"

ENTITY_PROFILES = f"{DATA_DIR}/entity_profiles.csv"

In [4]:
%cd /content/drive/MyDrive/BehavioralAI

/content/drive/MyDrive/BehavioralAI


In [5]:
import sklearn

import numpy as np
import pandas as pd
# import torch
print(sklearn.__version__)

1.6.1


In [6]:
!pip install -r requirements-colab.txt --prefer-binary

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!pip install river

In [ ]:
import pandas as pd
# import pyod

logs = pd.read_parquet("/content/drive/MyDrive/BehavioralAI/data/access_logs.parquet")

logs.head()

In [9]:
"""
attack_pipeline.py
===================
End-to-end glue: generates/loads data with `data_generator.py`, runs the
IForest/HalfSpaceTrees baseline (`baseline_profiling.HybridBehavioralProfiler`)
event-by-event, builds sliding windows and trains the Bi-LSTM
(`sequence_model.py`), joins both signals with session metadata via
`attack_classifier.AttackFeatureBuilder`, and trains the final XGBoost
multi-class classifier (`attack_classifier.XGBoostAttackClassifier`).

Run standalone for a demo:
    python attack_pipeline.py --days 30 --num-users 200

Or import `AttackClassificationPipeline` and call `.fit(logs_df)` /
`.predict(logs_df)` from your own training/inference code (e.g. a
Prefect/Airflow job, or the streaming consumer that writes into
`processed_streaming_logs`).
"""

from __future__ import annotations

import argparse
import ast
import json
import os
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from attack_classifier import AttackFeatureBuilder, XGBoostAttackClassifier
from baseline_profiling import FeatureVector, HybridBehavioralProfiler
from sequence_model import (
    BiLSTMSequenceModel,
    BiLSTMTrainer,
    SequenceWindowBuilder,
    Vocabulary,
    WindowDataset,
)

# Filenames used by both `.save()` / `.load_pretrained()` here and by
# `anomaly_pipeline.AnomalyDetectionPipeline`, so both modules agree on
# the on-disk layout of a `model_dir`.
BILSTM_FILENAME = "bilstm.pt"
XGB_FILENAME = "xgb_attack_classifier.joblib"
PROFILER_FILENAME = "behavioral_profiler.joblib"
PROFILES_FILENAME = "entity_profiles.csv"
METADATA_FILENAME = "pipeline_metadata.json"


def _dedupe_join_keys(logs_df: pd.DataFrame) -> pd.DataFrame:
    """`_score_baseline`, the Bi-LSTM window builder, and
    `AttackFeatureBuilder.from_dataframe` all join intermediate results
    back onto the raw events via (entity_id, timestamp). Bursty
    synthetic attacks (brute_force, credential_stuffing) can produce
    several events for the same entity at the exact same timestamp,
    which turns those joins into fan-outs instead of 1:1 matches. Nudge
    duplicates by a stable, order-preserving microsecond offset so
    (entity_id, timestamp) is always unique -- inconsequential for every
    downstream feature (hour-of-day, session duration, sequence order)
    but keeps every join exact."""
    df = logs_df.sort_values(["entity_id", "timestamp"], kind="stable").reset_index(drop=True)
    dup_rank = df.groupby(["entity_id", "timestamp"]).cumcount()
    if dup_rank.max() > 0:
        df["timestamp"] = df["timestamp"] + pd.to_timedelta(dup_rank, unit="us")
    return df


def _haversine_km(lat1, lon1, lat2, lon2) -> float:
    from math import asin, cos, radians, sin, sqrt
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 6371.0 * 2 * asin(sqrt(a))


class AttackClassificationPipeline:
    """Owns the three model layers and the joins between them.

    Typical usage
    -------------
        pipeline = AttackClassificationPipeline(entity_profiles_df)
        pipeline.fit(raw_logs_df)                 # trains Bi-LSTM + XGBoost
        preds = pipeline.predict(new_logs_df)      # per-event attack label + probs
    """

    def __init__(
        self,
        entity_profiles_df: pd.DataFrame,
        window_size: int = 10,
        bilstm_epochs: int = 8,
        random_state: int = 42,
    ):
        self.entity_profiles = entity_profiles_df.set_index("entity_id", drop=False)
        self.window_size = window_size
        self.bilstm_epochs = bilstm_epochs
        self.random_state = random_state

        self.resource_vocab, self.command_vocab = Vocabulary.from_generator_pools()
        self.window_builder = SequenceWindowBuilder(
            self.resource_vocab, self.command_vocab, window_size=window_size,
        )
        self.behavioral_profiler = HybridBehavioralProfiler(random_state=random_state)
        self.bilstm_trainer: Optional[BiLSTMTrainer] = None
        self.xgb_classifier = XGBoostAttackClassifier(random_state=random_state)

    # -- stage 1: IForest / online baseline, scored event-by-event -------
    def _score_baseline(self, logs_df: pd.DataFrame) -> pd.DataFrame:
        """Streams every event through `HybridBehavioralProfiler.observe()`
        in timestamp order (required -- it's a stateful online model) and
        returns a (entity_id, timestamp, iforest_score, iforest_is_anomaly)
        table to join back onto the raw logs."""
        df = logs_df.sort_values("timestamp").reset_index(drop=True)
        rows = []
        for _, row in df.iterrows():
            profile = self.entity_profiles.loc[row["entity_id"]] if row["entity_id"] in self.entity_profiles.index else None
            geo_km = 0.0
            if profile is not None:
                geo = row["geo_location"]
                lat = geo["lat"] if isinstance(geo, dict) else row.get("geo_lat")
                lon = geo["lon"] if isinstance(geo, dict) else row.get("geo_lon")
                if lat is not None and lon is not None:
                    geo_km = _haversine_km(lat, lon, profile["home_lat"], profile["home_lon"])

            cmds = row.get("command_sequence")
            if cmds is None or (isinstance(cmds, float) and pd.isna(cmds)):
                cmds = []
            elif isinstance(cmds, np.ndarray):
                cmds = cmds.tolist()
            fv = FeatureVector(
                login_hour=float(row["timestamp"].hour),
                session_duration=float(row["session_duration"]),
                geo_distance_km=float(geo_km),
                failure_count=1.0 if row.get("auth_result") != "success" else 0.0,
                extra={"num_commands": float(len(cmds))},
            )
            result = self.behavioral_profiler.observe(row["entity_id"], fv)
            rows.append({
                "entity_id": row["entity_id"],
                "timestamp": row["timestamp"],
                "iforest_score": result.combined_score,
                "iforest_is_anomaly": result.is_anomaly,
                "geo_distance_km": geo_km,
            })
        return pd.DataFrame(rows)

    # -- stage 2: Bi-LSTM sequence features -------------------------------
    def _fit_bilstm(self, logs_df: pd.DataFrame) -> pd.DataFrame:
        batch = self.window_builder.build(logs_df)
        dataset = WindowDataset(batch)
        n = len(dataset)
        val_n = max(1, int(0.15 * n))
        train_set, val_set = torch.utils.data.random_split(
            dataset, [n - val_n, val_n],
            generator=torch.Generator().manual_seed(self.random_state),
        )
        train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=256, shuffle=False)

        model = BiLSTMSequenceModel(
            num_resources=len(self.resource_vocab), num_commands=len(self.command_vocab),
        )
        class_weights = BiLSTMTrainer.compute_class_weights(batch.label_ids)
        self.bilstm_trainer = BiLSTMTrainer(model, class_weights=class_weights)
        self.bilstm_trainer.train(train_loader, epochs=self.bilstm_epochs, val_loader=val_loader)

        seq_features = self.bilstm_trainer.extract_features(batch)
        return pd.DataFrame({
            "entity_id": seq_features.entity_ids,
            "timestamp": seq_features.end_timestamps,
            "bilstm_sequence_loss": seq_features.sequence_loss,
            "bilstm_normal_prob": seq_features.normal_probability,
        })

    def _bilstm_infer(self, logs_df: pd.DataFrame) -> pd.DataFrame:
        if self.bilstm_trainer is None:
            raise RuntimeError("Bi-LSTM not trained yet -- call fit() first.")
        batch = self.window_builder.build(logs_df)
        seq_features = self.bilstm_trainer.extract_features(batch)
        return pd.DataFrame({
            "entity_id": seq_features.entity_ids,
            "timestamp": seq_features.end_timestamps,
            "bilstm_sequence_loss": seq_features.sequence_loss,
            "bilstm_normal_prob": seq_features.normal_probability,
        })

    # -- public API ---------------------------------------------------------
    def fit(self, logs_df: pd.DataFrame) -> Dict[str, object]:
        logs_df = _dedupe_join_keys(logs_df)
        logs_df = logs_df.merge(
            self.entity_profiles["entity_type"].rename("entity_type_lookup"),
            left_on="entity_id", right_index=True, how="left",
        )
        logs_df["entity_type"] = logs_df["entity_type"].fillna(logs_df.pop("entity_type_lookup"))

        iforest_scores = self._score_baseline(logs_df)
        bilstm_features = self._fit_bilstm(logs_df)
        features_df = AttackFeatureBuilder.from_dataframe(logs_df, iforest_scores, bilstm_features)
        return self.xgb_classifier.train(features_df)

    def predict(self, logs_df: pd.DataFrame) -> pd.DataFrame:
        logs_df = _dedupe_join_keys(logs_df)
        iforest_scores = self._score_baseline(logs_df)
        bilstm_features = self._bilstm_infer(logs_df)
        features_df = AttackFeatureBuilder.from_dataframe(logs_df, iforest_scores, bilstm_features)
        preds = self.xgb_classifier.predict(features_df)
        probs = self.xgb_classifier.predict_proba(features_df)
        out = logs_df[["entity_id", "timestamp"]].copy()
        out["predicted_label"] = preds
        out = pd.concat([out.reset_index(drop=True), probs.reset_index(drop=True)], axis=1)
        return out

    # -- persistence -----------------------------------------------------
    def save(self, model_dir: str) -> None:
        """Persist every trained artifact needed to run inference later
        with zero retraining:
          - `bilstm.pt`    -- Bi-LSTM weights + architecture config +
                              the resource/command vocabularies (the
                              embedding tables are meaningless without
                              the exact vocab that produced their ids).
          - `xgb_attack_classifier.joblib` -- the trained XGBoost model.
          - `behavioral_profiler.joblib`   -- the *entire* fitted
                              IForest/River baseline state, so a loaded
                              pipeline picks up exactly where training
                              left off instead of cold-starting.
          - `entity_profiles.csv`          -- the entity baselines used
                              for geo-distance features, so callers
                              don't have to keep the original DataFrame
                              around just to call `.load_pretrained()`.
          - `pipeline_metadata.json`       -- the constructor args
                              (window_size, random_state, ...) needed to
                              reconstruct non-model pipeline state.
        Raises if `.fit()` hasn't been called yet (nothing to save).
        """
        if self.bilstm_trainer is None:
            raise RuntimeError("Nothing to save -- call fit() before save().")

        os.makedirs(model_dir, exist_ok=True)

        self.bilstm_trainer.save(
            os.path.join(model_dir, BILSTM_FILENAME),
            resource_vocab=self.resource_vocab,
            command_vocab=self.command_vocab,
        )
        self.xgb_classifier.save(os.path.join(model_dir, XGB_FILENAME))
        self.behavioral_profiler.save(os.path.join(model_dir, PROFILER_FILENAME))
        self.entity_profiles.to_csv(os.path.join(model_dir, PROFILES_FILENAME), index=False)

        with open(os.path.join(model_dir, METADATA_FILENAME), "w") as f:
            json.dump({"window_size": self.window_size, "random_state": self.random_state}, f)

        print(f"Saved pipeline artifacts to {model_dir}/ "
              f"({BILSTM_FILENAME}, {XGB_FILENAME}, {PROFILER_FILENAME}, {PROFILES_FILENAME}, {METADATA_FILENAME})")

    @classmethod
    def load_pretrained(
        cls,
        model_dir: str,
        entity_profiles_df: Optional[pd.DataFrame] = None,
        device: Optional[str] = None,
    ) -> "AttackClassificationPipeline":
        """Reconstruct a fully trained pipeline straight from a
        `model_dir` written by `.save()` -- no `.fit()` / training loop
        involved. `entity_profiles_df` is only needed if the saved
        `entity_profiles.csv` isn't available or you want to score
        against a different/updated entity roster than what was trained
        on (e.g. new entities onboarded since training)."""
        bilstm_path = os.path.join(model_dir, BILSTM_FILENAME)
        xgb_path = os.path.join(model_dir, XGB_FILENAME)
        profiler_path = os.path.join(model_dir, PROFILER_FILENAME)
        profiles_path = os.path.join(model_dir, PROFILES_FILENAME)
        metadata_path = os.path.join(model_dir, METADATA_FILENAME)

        for required in (bilstm_path, xgb_path):
            if not os.path.exists(required):
                raise FileNotFoundError(f"Expected {required} -- has this pipeline been saved with .save()?")

        bilstm_trainer, resource_vocab, command_vocab = BiLSTMTrainer.load_pretrained(bilstm_path, device=device)
        if resource_vocab is None or command_vocab is None:
            raise ValueError(
                f"{bilstm_path} was saved without vocabularies -- can't safely run inference "
                "(resource/command token ids would be undefined). Re-save via AttackClassificationPipeline.save()."
            )

        xgb_classifier = XGBoostAttackClassifier.load(xgb_path)

        if entity_profiles_df is None:
            if not os.path.exists(profiles_path):
                raise FileNotFoundError(
                    f"No entity_profiles_df supplied and {profiles_path} not found -- "
                    "pass entity_profiles_df explicitly."
                )
            entity_profiles_df = pd.read_csv(profiles_path)
            entity_profiles_df["typical_resources"] = entity_profiles_df["typical_resources"].apply(ast.literal_eval)

        behavioral_profiler = (
            HybridBehavioralProfiler.load(profiler_path)
            if os.path.exists(profiler_path)
            else HybridBehavioralProfiler()
        )

        metadata = {}
        if os.path.exists(metadata_path):
            with open(metadata_path) as f:
                metadata = json.load(f)
        window_size = metadata.get("window_size", 10)
        random_state = metadata.get("random_state", 42)

        obj = cls.__new__(cls)  # bypass __init__ -- we're wiring up loaded components, not building fresh ones
        obj.entity_profiles = entity_profiles_df.set_index("entity_id", drop=False)
        obj.window_size = window_size
        obj.bilstm_epochs = 0  # not applicable -- this pipeline was loaded, not trained, in this process
        obj.random_state = random_state
        obj.resource_vocab = resource_vocab
        obj.command_vocab = command_vocab
        obj.window_builder = SequenceWindowBuilder(resource_vocab, command_vocab, window_size=window_size)
        obj.behavioral_profiler = behavioral_profiler
        obj.bilstm_trainer = bilstm_trainer
        obj.xgb_classifier = xgb_classifier
        return obj


def models_exist(model_dir: str) -> bool:
    """True if `model_dir` contains a complete enough set of artifacts
    for `AttackClassificationPipeline.load_pretrained()` to succeed
    (the two files it hard-requires; profiles/profiler/metadata all
    have fallbacks)."""
    return all(
        os.path.exists(os.path.join(model_dir, fname))
        for fname in (BILSTM_FILENAME, XGB_FILENAME)
    )


def main():
    parser = argparse.ArgumentParser(
        description="Train (or load pretrained) Bi-LSTM + XGBoost attack classification stack on synthetic data, "
                     "then run inference on a fresh batch and print a held-out/streaming report."
    )
    parser.add_argument("--num-users", type=int, default=100)
    parser.add_argument("--num-service-accounts", type=int, default=20)
    parser.add_argument("--num-devices", type=int, default=30)
    parser.add_argument("--days", type=int, default=14)
    parser.add_argument("--predict-days", type=int, default=2, help="Days of fresh data to run inference on after training/loading")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--bilstm-epochs", type=int, default=8)
    parser.add_argument("--model-out-dir", type=str, default="models")
    parser.add_argument("--force-retrain", action="store_true", help="Retrain even if models already has saved artifacts")
    args, _ = parser.parse_known_args()

    from datetime import datetime, timedelta

    from data_generator import ATTACK_RATE_RANGE, SyntheticDataGenerator

    gen = SyntheticDataGenerator(
        num_users=args.num_users, num_service_accounts=args.num_service_accounts,
        num_devices=args.num_devices, seed=args.seed,
    )
    profiles_df = gen.profiles_dataframe()

    if models_exist(args.model_out_dir) and not args.force_retrain:
        print(f"Found existing model artifacts in {args.model_out_dir}/ -- loading pretrained pipeline (no training).")
        pipeline = AttackClassificationPipeline.load_pretrained(args.model_out_dir, entity_profiles_df=profiles_df)
    else:
        print(f"No usable model artifacts in {args.model_out_dir}/ -- training from scratch.")
        start = datetime.utcnow() - timedelta(days=args.days + args.predict_days)
        train_df = gen.generate(start, args.days, attack_rate_range=ATTACK_RATE_RANGE)

        pipeline = AttackClassificationPipeline(profiles_df, bilstm_epochs=args.bilstm_epochs, random_state=args.seed)
        metrics = pipeline.fit(train_df)
        print("Held-out classification report:")
        print(metrics["classification_report"])

        pipeline.save(args.model_out_dir)

    # Either way (loaded or freshly trained), run inference on a fresh
    # batch to demonstrate `predict()` works standalone off the pipeline
    # we ended up with.
    print(f"\nRunning inference on {args.predict_days} fresh day(s) of synthetic data...")
    predict_start = datetime.utcnow() - timedelta(days=args.predict_days)
    predict_df = gen.generate(predict_start, args.predict_days, attack_rate_range=ATTACK_RATE_RANGE)
    predictions = pipeline.predict(predict_df)

    predicted_attacks = predictions[predictions["predicted_label"] != "normal"]
    print(f"Scored {len(predictions)} events -- {len(predicted_attacks)} flagged as an attack category.")
    if len(predicted_attacks):
        print(predicted_attacks["predicted_label"].value_counts())


if __name__ == "__main__":
    main()

No usable model artifacts in models/ -- training from scratch.


/tmp/ipykernel_13675/1417893740.py:368: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow() - timedelta(days=args.days + args.predict_days)


[SyntheticDataGenerator] entities=150 normal_events=5245 anomalous_events=895 total=6140 anomaly_rate=14.58%
label
normal                       5245
lateral_movement              178
brute_force                   168
low_and_slow_exfiltration     156
device_spoofing               127
insider_drift                 100
credential_stuffing            86
impossible_travel              80
Name: count, dtype: int64
[BiLSTM] epoch 1/8 train_loss=1.9759 val_loss=1.7263
[BiLSTM] epoch 2/8 train_loss=1.4895 val_loss=1.2493
[BiLSTM] epoch 3/8 train_loss=1.1057 val_loss=1.0391
[BiLSTM] epoch 4/8 train_loss=0.8282 val_loss=0.8796
[BiLSTM] epoch 5/8 train_loss=0.6548 val_loss=0.7833
[BiLSTM] epoch 6/8 train_loss=0.5628 val_loss=0.7455
[BiLSTM] epoch 7/8 train_loss=0.5015 val_loss=0.6974
[BiLSTM] epoch 8/8 train_loss=0.4224 val_loss=0.6950
                           precision    recall  f1-score   support

                   normal       1.00      1.00      1.00      1049
              brute_force   

/tmp/ipykernel_13675/1417893740.py:382: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  predict_start = datetime.utcnow() - timedelta(days=args.predict_days)


Scored 1055 events -- 163 flagged as an attack category.
predicted_label
credential_stuffing          66
device_spoofing              29
lateral_movement             25
brute_force                  23
impossible_travel            13
low_and_slow_exfiltration     4
insider_drift                 3
Name: count, dtype: int64


In [11]:
import joblib

obj = joblib.load("models/xgb_attack_classifier.joblib")
print(type(obj))

<class 'dict'>


In [ ]:
"""
sequence_model.py
==================
Sequential / temporal modeling layer for the Behavioral Anomaly Detection
system: a Bi-LSTM that ingests sliding windows of an entity's recent
events (resource accessed + command issued + light numeric context per
event) and learns to recognize what a "normal" trajectory through that
space looks like.

Why this exists alongside the Isolation Forest baseline (baseline_profiling.py)
--------------------------------------------------------------------------
The IForest/HalfSpaceTrees baseline in `baseline_profiling.py` scores each
event *independently* -- it has no notion of order. That is exactly the
blind spot attacks like `lateral_movement` (rapid escalating hops across
resources) and `low_and_slow_exfiltration` (sparse pulls spread across
days, each individually unremarkable) are built to exploit. The Bi-LSTM
here looks at *sequences* of events per entity and produces a per-window
"sequence loss" -- how surprising this trajectory is under the model's
learned notion of normal -- which becomes one of the input features to
the downstream XGBoost multi-class attack classifier in
`attack_classifier.py`.

Pipeline
--------
1. `Vocabulary`            -- token <-> id maps for resources and commands.
2. `SequenceWindowBuilder` -- turns a raw_access_logs-shaped DataFrame
   into fixed-length, left-padded sliding windows per entity.
3. `WindowDataset`         -- torch Dataset wrapping the built windows.
4. `BiLSTMSequenceModel`   -- embeds resource/command tokens, concatenates
   numeric context, runs a bidirectional LSTM, and predicts a window-level
   label distribution (normal vs. the 7 attack types).
5. `BiLSTMTrainer`         -- training loop, evaluation, and the
   `extract_features()` inference method that produces the per-window
   "sequence loss" / anomaly-probability features consumed downstream.

Integrates with
----------------
- `models.py`     : `LabelType`, `EntityType` enums (window labels map
                    1:1 onto `LabelType`; label ids follow `LABEL_ORDER`
                    below, the same order used by `attack_classifier.py`).
- `data_generator.py` : `BENIGN_COMMANDS`, `PRIV_ESC_COMMANDS`,
                    `EXFIL_COMMANDS`, `RESOURCE_POOL` seed the vocabulary
                    so token ids are stable across train/inference even
                    for resources/commands not yet observed for a given
                    entity.
- `baseline_profiling.py` : consumed alongside (not by) this module --
                    `attack_classifier.py` is what joins IForest scores
                    with the sequence features produced here.
"""


In [ ]:

from __future__ import annotations

import argparse
import ast
import json
import os
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from attack_classifier import AttackFeatureBuilder, XGBoostAttackClassifier
from baseline_profiling import FeatureVector, HybridBehavioralProfiler
from sequence_model import (
    BiLSTMSequenceModel,
    BiLSTMTrainer,
    SequenceWindowBuilder,
    Vocabulary,
    WindowDataset,
)


In [ ]:
    

# Run directly in the notebook using the existing logs/profiles DataFrames
train_logs_df = globals().get("logs_df")
train_profiles_df = globals().get("profiles_df")

if train_logs_df is None or train_profiles_df is None:
    raise NameError("Expected logs_df and profiles_df to be available in the notebook environment.")

pipeline = AttackClassificationPipeline(
    train_profiles_df,
    bilstm_epochs=8,
    random_state=42,
)
metrics = pipeline.fit(train_logs_df)
print(metrics["classification_report"])


In [ ]:

try:
    from models import EntityType, LabelType
except ImportError:  # pragma: no cover - standalone use
    import enum

    class EntityType(str, enum.Enum):
        user = "user"
        service_account = "service_account"
        edge_device = "edge_device"

    class LabelType(str, enum.Enum):
        normal = "normal"
        brute_force = "brute_force"
        impossible_travel = "impossible_travel"
        credential_stuffing = "credential_stuffing"
        lateral_movement = "lateral_movement"
        device_spoofing = "device_spoofing"
        low_and_slow_exfiltration = "low_and_slow_exfiltration"
        insider_drift = "insider_drift"

from sequence_model import LABEL_ORDER, LABEL_TO_ID

ENTITY_TYPE_ORDER: List[str] = [e.value for e in EntityType]

SENSITIVE_SUBSTRINGS = ("finance", "customer_db", "billing")

FEATURE_COLUMNS: List[str] = [
    "iforest_score",
    "iforest_is_anomaly",
    "bilstm_sequence_loss",
    "bilstm_normal_prob",
    "session_duration",
    "hour_sin",
    "hour_cos",
    "is_auth_failure",
    "num_commands",
    "is_sensitive_resource",
    "geo_distance_km",
] + [f"entity_type_{t}" for t in ENTITY_TYPE_ORDER]


In [ ]:

# ---------------------------------------------------------------------------
# Feature assembly
# ---------------------------------------------------------------------------
@dataclass
class EventContext:
    """One row of raw signal to be turned into a feature vector.

    `iforest_score` / `iforest_is_anomaly` come from
    `HybridBehavioralProfiler.observe(...)` (see `baseline_profiling.py`).
    `bilstm_sequence_loss` / `bilstm_normal_prob` come from
    `BiLSTMTrainer.extract_features(...)` (see `sequence_model.py`),
    joined on (entity_id, timestamp).
    """

    entity_id: str
    entity_type: str
    timestamp: pd.Timestamp
    session_duration: float
    auth_result: str
    command_sequence: List[str]
    resource_accessed: str
    geo_distance_km: float
    iforest_score: float
    iforest_is_anomaly: bool
    bilstm_sequence_loss: float
    bilstm_normal_prob: float
    label: Optional[str] = None


class AttackFeatureBuilder:
    """Joins IForest + Bi-LSTM signals with session metadata into the
    flat feature table XGBoost trains/predicts on."""

    def build(self, contexts: List[EventContext]) -> pd.DataFrame:
        rows = []
        for ctx in contexts:
            hour = ctx.timestamp.hour + ctx.timestamp.minute / 60.0
            row: Dict[str, float] = {
                "iforest_score": float(ctx.iforest_score),
                "iforest_is_anomaly": float(bool(ctx.iforest_is_anomaly)),
                "bilstm_sequence_loss": float(ctx.bilstm_sequence_loss),
                "bilstm_normal_prob": float(ctx.bilstm_normal_prob),
                "session_duration": float(ctx.session_duration),
                "hour_sin": math.sin(2 * math.pi * hour / 24.0),
                "hour_cos": math.cos(2 * math.pi * hour / 24.0),
                "is_auth_failure": 1.0 if ctx.auth_result != "success" else 0.0,
                "num_commands": float(len(ctx.command_sequence or [])),
                "is_sensitive_resource": 1.0 if any(s in ctx.resource_accessed for s in SENSITIVE_SUBSTRINGS) else 0.0,
                "geo_distance_km": float(ctx.geo_distance_km),
            }
            for t in ENTITY_TYPE_ORDER:
                row[f"entity_type_{t}"] = 1.0 if ctx.entity_type == t else 0.0
            if ctx.label is not None:
                row["label"] = ctx.label
                row["_entity_id"] = ctx.entity_id
            rows.append(row)

        df = pd.DataFrame(rows)
        # Guarantee stable column order/presence even if a batch happens
        # to omit a class of entity_type etc.
        for col in FEATURE_COLUMNS:
            if col not in df.columns:
                df[col] = 0.0
        return df

    @staticmethod
    def from_dataframe(
        events_df: pd.DataFrame,
        iforest_scores: pd.DataFrame,
        bilstm_features: pd.DataFrame,
    ) -> pd.DataFrame:
        """Convenience path used by `attack_pipeline.py`: join three
        already-computed tables on (entity_id, timestamp) instead of
        constructing `EventContext` objects one at a time.

        - events_df: raw_access_logs-shaped rows (entity_id, entity_type,
          timestamp, session_duration, auth_result, command_sequence,
          resource_accessed, geo_lat/geo_lon or geo_distance_km, label)
        - iforest_scores: columns [entity_id, timestamp, iforest_score, iforest_is_anomaly]
        - bilstm_features: columns [entity_id, timestamp, bilstm_sequence_loss, bilstm_normal_prob]
        """
        merged = events_df.merge(iforest_scores, on=["entity_id", "timestamp"], how="left")
        merged = merged.merge(bilstm_features, on=["entity_id", "timestamp"], how="left")
        merged[["iforest_score", "iforest_is_anomaly", "bilstm_sequence_loss", "bilstm_normal_prob"]] = (
            merged[["iforest_score", "iforest_is_anomaly", "bilstm_sequence_loss", "bilstm_normal_prob"]].fillna(0.0)
        )
        if "geo_distance_km" not in merged.columns:
            merged["geo_distance_km"] = 0.0

        builder = AttackFeatureBuilder()
        contexts = [
            EventContext(
                entity_id=r["entity_id"],
                entity_type=r["entity_type"],
                timestamp=r["timestamp"],
                session_duration=r.get("session_duration", 0.0),
                auth_result=r.get("auth_result", "success"),
                command_sequence=r.get("command_sequence", []) or [],
                resource_accessed=r.get("resource_accessed", ""),
                geo_distance_km=r.get("geo_distance_km", 0.0),
                iforest_score=r["iforest_score"],
                iforest_is_anomaly=bool(r["iforest_is_anomaly"]),
                bilstm_sequence_loss=r["bilstm_sequence_loss"],
                bilstm_normal_prob=r["bilstm_normal_prob"],
                label=r.get("label"),
            )
            for _, r in merged.iterrows()
        ]
        return builder.build(contexts)


In [ ]:

# ---------------------------------------------------------------------------
# XGBoost classifier
# ---------------------------------------------------------------------------
class XGBoostAttackClassifier:
    """Multi-class (8-way) attack classifier on top of assembled features.

    Label ids follow `sequence_model.LABEL_ORDER` (0 = normal), so this
    classifier's predictions line up 1:1 with the Bi-LSTM's own label
    space and with `models.LabelType`.
    """

    def __init__(
        self,
        num_classes: int = len(LABEL_ORDER),
        max_depth: int = 6,
        n_estimators: int = 300,
        learning_rate: float = 0.08,
        random_state: int = 42,
        **xgb_kwargs,
    ):
        self.feature_columns = FEATURE_COLUMNS
        self.model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=num_classes,
            max_depth=max_depth,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            random_state=random_state,
            **xgb_kwargs,
        )
        self.fitted = False

    def _sample_weights(self, y: np.ndarray) -> np.ndarray:
        """Inverse-frequency sample weights -- attacks are 0.5-3% of
        volume each, so unweighted training would just predict 'normal'."""
        counts = np.bincount(y, minlength=len(LABEL_ORDER)).astype(np.float64)
        counts[counts == 0] = 1.0
        class_weight = counts.sum() / (len(LABEL_ORDER) * counts)
        return class_weight[y]

    def train(
        self,
        features_df: pd.DataFrame,
        label_col: str = "label",
        test_size: float = 0.2,
        random_state: int = 42,
        verbose: bool = True,
    ) -> Dict[str, object]:
        """Train on an `AttackFeatureBuilder` output that also has the
        ground-truth `label` (string) column. Returns a dict with a held-out
        classification report and confusion matrix for quick sanity-checking."""
        X = features_df[self.feature_columns].values
        y = features_df[label_col].map(LABEL_TO_ID).values

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        sw_train = self._sample_weights(y_train)

        self.model.fit(X_train, y_train, sample_weight=sw_train)
        self.fitted = True

        y_pred = self.model.predict(X_test)
        report = classification_report(
            y_test, y_pred, labels=list(range(len(LABEL_ORDER))),
            target_names=LABEL_ORDER, output_dict=True, zero_division=0,
        )
        cm = confusion_matrix(y_test, y_pred, labels=list(range(len(LABEL_ORDER))))

        if verbose:
            print(classification_report(
                y_test, y_pred, labels=list(range(len(LABEL_ORDER))),
                target_names=LABEL_ORDER, zero_division=0,
            ))

        return {"classification_report": report, "confusion_matrix": cm}

    def predict(self, features_df: pd.DataFrame) -> np.ndarray:
        """Returns predicted `LabelType` string per row."""
        self._check_fitted()
        X = features_df[self.feature_columns].values
        pred_ids = self.model.predict(X)
        return np.array([LABEL_ORDER[i] for i in pred_ids])

    def predict_proba(self, features_df: pd.DataFrame) -> pd.DataFrame:
        self._check_fitted()
        X = features_df[self.feature_columns].values
        probs = self.model.predict_proba(X)
        return pd.DataFrame(probs, columns=LABEL_ORDER, index=features_df.index)

    def feature_importances(self) -> pd.Series:
        self._check_fitted()
        return pd.Series(self.model.feature_importances_, index=self.feature_columns).sort_values(ascending=False)

    def _check_fitted(self) -> None:
        if not self.fitted:
            raise RuntimeError("XGBoostAttackClassifier must be trained (or loaded) before predicting.")

    def save(self, path: str) -> None:
        joblib.dump({"model": self.model, "feature_columns": self.feature_columns}, path)

    @classmethod
    def load(cls, path: str) -> "XGBoostAttackClassifier":
        payload = joblib.load(path)
        obj = cls.__new__(cls)
        obj.model = payload["model"]
        obj.feature_columns = payload["feature_columns"]
        obj.fitted = True
        return obj


In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset

In [ ]:

try:
    # Ground-truth label enum + a stable class ordering shared with
    # attack_classifier.py so both modules agree on label <-> id mapping.
    from models import LabelType
except ImportError:  # pragma: no cover - allows standalone use/testing
    import enum

    class LabelType(str, enum.Enum):
        normal = "normal"
        brute_force = "brute_force"
        impossible_travel = "impossible_travel"
        credential_stuffing = "credential_stuffing"
        lateral_movement = "lateral_movement"
        device_spoofing = "device_spoofing"
        low_and_slow_exfiltration = "low_and_slow_exfiltration"
        insider_drift = "insider_drift"


In [ ]:

# Fixed class ordering used everywhere in the sequence + classification
# stack. Index 0 is always "normal".
LABEL_ORDER: List[str] = [
    LabelType.normal.value,
    LabelType.brute_force.value,
    LabelType.impossible_travel.value,
    LabelType.credential_stuffing.value,
    LabelType.lateral_movement.value,
    LabelType.device_spoofing.value,
    LabelType.low_and_slow_exfiltration.value,
    LabelType.insider_drift.value,
]
LABEL_TO_ID: Dict[str, int] = {name: i for i, name in enumerate(LABEL_ORDER)}
NORMAL_LABEL_ID = LABEL_TO_ID[LabelType.normal.value]

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


In [ ]:

# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------
class Vocabulary:
    """Simple token <-> id map with a reserved PAD (id 0) and UNK (id 1).

    Seed it with the known command/resource pools from `data_generator.py`
    at construction time so ids are stable regardless of what a given
    training slice happens to contain; anything unseen at inference falls
    back to UNK rather than crashing or silently reassigning ids.
    """

    def __init__(self, tokens: Optional[Sequence[str]] = None):
        self.token_to_id: Dict[str, int] = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.id_to_token: List[str] = [PAD_TOKEN, UNK_TOKEN]
        if tokens:
            self.add_many(tokens)

    def add(self, token: str) -> int:
        if token not in self.token_to_id:
            self.token_to_id[token] = len(self.id_to_token)
            self.id_to_token.append(token)
        return self.token_to_id[token]

    def add_many(self, tokens: Sequence[str]) -> None:
        for t in tokens:
            self.add(t)

    def encode(self, token: Optional[str]) -> int:
        if token is None:
            return self.token_to_id[PAD_TOKEN]
        return self.token_to_id.get(token, self.token_to_id[UNK_TOKEN])

    def __len__(self) -> int:
        return len(self.id_to_token)

    @classmethod
    def from_generator_pools(cls) -> Tuple["Vocabulary", "Vocabulary"]:
        """Build (resource_vocab, command_vocab) seeded from data_generator's
        reference pools, falling back to a small built-in set if that
        module isn't importable (e.g. this file used standalone)."""
        try:
            from data_generator import COMMANDS_POOL, RESOURCE_POOL
            resources, commands = RESOURCE_POOL, COMMANDS_POOL
        except ImportError:
            resources = [f"resource/generic/{i:03d}" for i in range(50)]
            commands = ["ls", "cat", "whoami", "sudo", "scp", "curl", "kubectl"]
        return cls(resources), cls(commands)



In [ ]:


# ---------------------------------------------------------------------------
# Windowing
# ---------------------------------------------------------------------------
NUMERIC_FEATURE_NAMES = [
    "hour_sin", "hour_cos", "session_duration_z", "is_failure",
    "num_commands", "is_sensitive_resource",
]


@dataclass
class WindowBatch:
    """Container of encoded, padded sliding windows ready for the model."""

    entity_ids: List[str]
    end_timestamps: List[pd.Timestamp]
    resource_ids: np.ndarray      # (N, W) int64
    command_ids: np.ndarray       # (N, W) int64
    numeric_feats: np.ndarray     # (N, W, F) float32
    lengths: np.ndarray           # (N,) int64 -- true (unpadded) length
    label_ids: np.ndarray         # (N,) int64 -- window-level label


class SequenceWindowBuilder:
    """Turns a raw_access_logs-shaped DataFrame into fixed-length,
    left-padded sliding windows, one per entity per step.

    Expected input columns (matching `RawAccessLog` / `data_generator`
    output): entity_id, timestamp, resource_accessed, command_sequence
    (list[str]), session_duration, auth_result, label.

    Window label = the label of the *last* event in the window (i.e. "is
    the most recent event, in the context of its recent history, part of
    an attack"). This matches how the model will be used at inference: at
    time T you have the last W events and want to judge the one that just
    happened.
    """

    def __init__(
        self,
        resource_vocab: Vocabulary,
        command_vocab: Vocabulary,
        window_size: int = 10,
        stride: int = 1,
        sensitive_resource_substrings: Sequence[str] = ("finance", "customer_db", "billing"),
    ):
        self.resource_vocab = resource_vocab
        self.command_vocab = command_vocab
        self.window_size = window_size
        self.stride = stride
        self.sensitive_substrings = sensitive_resource_substrings

    def _numeric_row(self, row: pd.Series, duration_mean: float, duration_std: float) -> List[float]:
        ts = row["timestamp"]
        hour = ts.hour + ts.minute / 60.0
        hour_sin = math.sin(2 * math.pi * hour / 24.0)
        hour_cos = math.cos(2 * math.pi * hour / 24.0)
        dur_z = (row["session_duration"] - duration_mean) / (duration_std or 1.0)
        is_failure = 1.0 if str(row.get("auth_result", "success")) != "success" else 0.0
        cmds = row.get("command_sequence") or []
        num_commands = float(len(cmds))
        resource = str(row["resource_accessed"])
        is_sensitive = 1.0 if any(s in resource for s in self.sensitive_substrings) else 0.0
        return [hour_sin, hour_cos, dur_z, is_failure, num_commands, is_sensitive]

    def build(self, logs_df: pd.DataFrame) -> WindowBatch:
        df = logs_df.sort_values(["entity_id", "timestamp"]).reset_index(drop=True)
        duration_mean = float(df["session_duration"].mean()) if len(df) else 0.0
        duration_std = float(df["session_duration"].std()) if len(df) else 1.0

        entity_ids: List[str] = []
        end_ts: List[pd.Timestamp] = []
        res_rows: List[List[int]] = []
        cmd_rows: List[List[int]] = []
        num_rows: List[List[List[float]]] = []
        lengths: List[int] = []
        label_rows: List[int] = []

        W = self.window_size
        for entity_id, group in df.groupby("entity_id", sort=False):
            group = group.reset_index(drop=True)
            n = len(group)
            for end in range(0, n, self.stride):
                start = max(0, end - W + 1)
                window = group.iloc[start:end + 1]
                true_len = len(window)

                res_ids = [self.resource_vocab.encode(str(r)) for r in window["resource_accessed"]]
                # primary command per event = first command in that event's
                # sequence (empty sequences -> PAD), a deliberate simplification
                # that keeps one token per timestep aligned with resource_ids.
                cmd_ids = [
                    self.command_vocab.encode(cmds[0]) if cmds else self.command_vocab.encode(None)
                    for cmds in window["command_sequence"]
                ]
                numeric = [self._numeric_row(r, duration_mean, duration_std) for _, r in window.iterrows()]

                pad_amt = W - true_len
                if pad_amt > 0:
                    res_ids = [0] * pad_amt + res_ids
                    cmd_ids = [0] * pad_amt + cmd_ids
                    numeric = [[0.0] * len(NUMERIC_FEATURE_NAMES)] * pad_amt + numeric

                entity_ids.append(entity_id)
                end_ts.append(window.iloc[-1]["timestamp"])
                res_rows.append(res_ids)
                cmd_rows.append(cmd_ids)
                num_rows.append(numeric)
                lengths.append(true_len)
                label_rows.append(LABEL_TO_ID.get(str(window.iloc[-1].get("label", "normal")), NORMAL_LABEL_ID))

        return WindowBatch(
            entity_ids=entity_ids,
            end_timestamps=end_ts,
            resource_ids=np.array(res_rows, dtype=np.int64) if res_rows else np.zeros((0, W), dtype=np.int64),
            command_ids=np.array(cmd_rows, dtype=np.int64) if cmd_rows else np.zeros((0, W), dtype=np.int64),
            numeric_feats=np.array(num_rows, dtype=np.float32) if num_rows else np.zeros((0, W, len(NUMERIC_FEATURE_NAMES)), dtype=np.float32),
            lengths=np.array(lengths, dtype=np.int64),
            label_ids=np.array(label_rows, dtype=np.int64),
        )


class WindowDataset(Dataset):
    """torch Dataset wrapping a `WindowBatch`."""

    def __init__(self, batch: WindowBatch):
        self.batch = batch

    def __len__(self) -> int:
        return len(self.batch.label_ids)

    def __getitem__(self, idx: int):
        b = self.batch
        return (
            torch.from_numpy(b.resource_ids[idx]),
            torch.from_numpy(b.command_ids[idx]),
            torch.from_numpy(b.numeric_feats[idx]),
            torch.tensor(b.label_ids[idx], dtype=torch.long),
        )



In [ ]:

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class BiLSTMSequenceModel(nn.Module):
    """Bidirectional LSTM over per-event (resource, command, numeric)
    tokens, pooled and classified into `LABEL_ORDER` classes.

    The classification head is trained on the synthetic ground-truth
    labels; at inference the *loss/negative-log-likelihood assigned to
    the "normal" class* (not the argmax label) is what's exposed as the
    "sequence loss" anomaly signal, since in production you don't have
    ground truth for new events -- see `BiLSTMTrainer.extract_features`.
    """

    def __init__(
        self,
        num_resources: int,
        num_commands: int,
        numeric_dim: int = len(NUMERIC_FEATURE_NAMES),
        resource_emb_dim: int = 32,
        command_emb_dim: int = 16,
        hidden_dim: int = 64,
        num_layers: int = 1,
        num_classes: int = len(LABEL_ORDER),
        dropout: float = 0.2,
    ):
        super().__init__()
        self.resource_emb = nn.Embedding(num_resources, resource_emb_dim, padding_idx=0)
        self.command_emb = nn.Embedding(num_commands, command_emb_dim, padding_idx=0)
        input_dim = resource_emb_dim + command_emb_dim + numeric_dim

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        # Attention pooling over the bi-directional hidden states, so the
        # model can weight the timestep(s) that matter most (e.g. the
        # privilege-escalation hop in a lateral-movement window) rather
        # than relying solely on the final hidden state.
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, resource_ids: torch.Tensor, command_ids: torch.Tensor, numeric_feats: torch.Tensor):
        r = self.resource_emb(resource_ids)
        c = self.command_emb(command_ids)
        x = torch.cat([r, c, numeric_feats], dim=-1)
        out, _ = self.lstm(x)               # (B, W, 2*hidden)
        out = self.dropout(out)
        attn_scores = torch.softmax(self.attn(out).squeeze(-1), dim=-1)  # (B, W)
        pooled = torch.einsum("bwh,bw->bh", out, attn_scores)            # (B, 2*hidden)
        logits = self.classifier(pooled)
        return logits, attn_scores


# ---------------------------------------------------------------------------
# Trainer / inference
# ---------------------------------------------------------------------------
@dataclass
class SequenceFeatures:
    """Per-window features handed off to the XGBoost classifier."""

    entity_ids: List[str]
    end_timestamps: List[pd.Timestamp]
    true_label_ids: np.ndarray
    predicted_label_ids: np.ndarray
    predicted_probs: np.ndarray            # (N, num_classes)
    sequence_loss: np.ndarray              # NLL assigned to "normal" class; higher = more anomalous
    normal_probability: np.ndarray         # P(normal); convenience = 1 - anomaly signal


class BiLSTMTrainer:
    """Training loop + evaluation + feature-extraction wrapper around
    `BiLSTMSequenceModel`."""

    def __init__(
        self,
        model: BiLSTMSequenceModel,
        lr: float = 1e-3,
        weight_decay: float = 1e-5,
        class_weights: Optional[torch.Tensor] = None,
        device: Optional[str] = None,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights.to(self.device) if class_weights is not None else None)

    @staticmethod
    def compute_class_weights(label_ids: np.ndarray, num_classes: int = len(LABEL_ORDER)) -> torch.Tensor:
        """Inverse-frequency class weights so the heavily-imbalanced
        normal/attack split doesn't collapse the model into always
        predicting 'normal'."""
        counts = np.bincount(label_ids, minlength=num_classes).astype(np.float64)
        counts[counts == 0] = 1.0
        weights = counts.sum() / (num_classes * counts)
        return torch.tensor(weights, dtype=torch.float32)

    def train(self, loader, epochs: int = 10, val_loader=None, verbose: bool = True) -> Dict[str, List[float]]:
        history: Dict[str, List[float]] = {"train_loss": [], "val_loss": []}
        for epoch in range(1, epochs + 1):
            self.model.train()
            running = 0.0
            for res, cmd, num, labels in loader:
                res, cmd, num, labels = (t.to(self.device) for t in (res, cmd, num, labels))
                self.optimizer.zero_grad()
                logits, _ = self.model(res, cmd, num)
                loss = self.criterion(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                self.optimizer.step()
                running += loss.item() * len(labels)
            train_loss = running / max(1, len(loader.dataset))
            history["train_loss"].append(train_loss)

            val_loss = None
            if val_loader is not None:
                val_loss = self.evaluate(val_loader)
                history["val_loss"].append(val_loss)

            if verbose:
                msg = f"[BiLSTM] epoch {epoch}/{epochs} train_loss={train_loss:.4f}"
                if val_loss is not None:
                    msg += f" val_loss={val_loss:.4f}"
                print(msg)
        return history

    @torch.no_grad()
    def evaluate(self, loader) -> float:
        self.model.eval()
        running = 0.0
        for res, cmd, num, labels in loader:
            res, cmd, num, labels = (t.to(self.device) for t in (res, cmd, num, labels))
            logits, _ = self.model(res, cmd, num)
            loss = self.criterion(logits, labels)
            running += loss.item() * len(labels)
        return running / max(1, len(loader.dataset))

    @torch.no_grad()
    def extract_features(self, batch: WindowBatch, batch_size: int = 512) -> SequenceFeatures:
        """Run inference over a `WindowBatch` and return the per-window
        features consumed by the XGBoost classifier: predicted class,
        full probability vector, and `sequence_loss` -- the negative
        log-likelihood the model assigns to the "normal" class for that
        window. A window whose recent trajectory looks nothing like
        normal behavior gets a high sequence_loss even if we don't know
        its true label, which is exactly what's needed at inference time
        in production (no ground truth available then).
        """
        self.model.eval()
        n = len(batch.label_ids)
        all_probs = np.zeros((n, len(LABEL_ORDER)), dtype=np.float32)

        for start in range(0, n, batch_size):
            end = min(n, start + batch_size)
            res = torch.from_numpy(batch.resource_ids[start:end]).to(self.device)
            cmd = torch.from_numpy(batch.command_ids[start:end]).to(self.device)
            num = torch.from_numpy(batch.numeric_feats[start:end]).to(self.device)
            logits, _ = self.model(res, cmd, num)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs[start:end] = probs

        eps = 1e-9
        normal_probability = all_probs[:, NORMAL_LABEL_ID]
        sequence_loss = -np.log(np.clip(normal_probability, eps, 1.0))
        predicted_label_ids = all_probs.argmax(axis=1)

        return SequenceFeatures(
            entity_ids=batch.entity_ids,
            end_timestamps=batch.end_timestamps,
            true_label_ids=batch.label_ids,
            predicted_label_ids=predicted_label_ids,
            predicted_probs=all_probs,
            sequence_loss=sequence_loss,
            normal_probability=normal_probability,
        )

    def save(self, path: str) -> None:
        torch.save(self.model.state_dict(), path)

    def load(self, path: str) -> None:
        self.model.load_state_dict(torch.load(path, map_location=self.device))


In [ ]:
model.save("/content/drive/MyDrive/model.joblib")